In [1]:
!pip install -U peft datasets sacrebleu sentencepiece accelerate evaluate bitsandbytes transformers==4.44.2 huggingface_hub==0.23.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of peft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of peft to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with

In [2]:
!pip uninstall peft -y
!pip install git+https://github.com/huggingface/peft.git

Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
  Cloning https://github.com/huggingface/peft.git to /tmp/pip-req-build-rl02qyjo
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/peft.git /tmp/pip-req-build-rl02qyjo
  Resolved https://github.com/huggingface/peft.git to commit 9a20e07d347fa5f1c565f3393aa0c4fea469badb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.5 MB/s eta 0:00:00
  Created wheel for peft: filename=peft-0.19.2.dev0-py3-none-any.whl size=678550 sha256=35e91d5c8a62ef2f1a4d1d3ef7976348c173ce9f5523e4ed36b824cd53478cd2
  Stored in directory: /tmp/pip-ephem-wheel-cache-h47l8rbf/wheels/5d/16/61/117d50be36b7cb532817817523554825ff840d223c0f65c2c4
Successfully built peft
  Attempting uninstall: huggingface_hub
    Found existing installati

In [3]:
import transformers
print(transformers.__version__)

4.44.2


In [4]:
from datasets import Dataset, DatasetDict, load_dataset
import random
import pandas as pd

# Load MedEV
ds = load_dataset("nhuvo/MedEV")
dataset = ds["train"]

offset = 340897
num_pairs = 340897

pairs = []

for i in range(num_pairs):
    en = dataset[i]["text"].strip()
    vi = dataset[i + offset]["text"].strip()

    # filter rác nhẹ
    if len(en) > 5 and len(vi) > 5:
        pairs.append({
            "en": en,
            "vi": vi
        })

print("Total pairs:", len(pairs))

README.md: 0.00B [00:00, ?B/s]

train.en.txt:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

train.vi.txt:   0%|          | 0.00/61.9M [00:00<?, ?B/s]

val.en.new.txt: 0.00B [00:00, ?B/s]

val.vi.new.txt: 0.00B [00:00, ?B/s]

...

test.vi.new.txt: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/681794 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17878 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17920 [00:00<?, ? examples/s]

Total pairs: 340752


In [5]:
print(pairs[1])

{'en': 'Evaluate clinical, subclinical symptoms of patients with otittis media effusion and V a at otorhinolaryngology department - Thai Nguyên National Hospital.', 'vi': 'Đánh giá đặc điểm lâm sàng, cận lâm sàng bệnh nhân viêm tai ứ dịch trên viêm V.a tại Khoa Tai mũi họng - Bệnh viện Trung ương Thái Nguyên.'}


In [6]:
random.seed(42)
random.shuffle(pairs)

pairs = pairs[:100000]  # lấy 100k

train_size = int(0.8 * len(pairs))
val_size   = int(0.19 * len(pairs))

train_pairs = pairs[:train_size]
val_pairs   = pairs[train_size:train_size+val_size]
test_pairs  = pairs[train_size+val_size:]

raw = DatasetDict({
    "train": Dataset.from_pandas(pd.DataFrame(train_pairs)),
    "validation": Dataset.from_pandas(pd.DataFrame(val_pairs)),
    "test": Dataset.from_pandas(pd.DataFrame(test_pairs)),
})

print(raw)

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 19000
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 1000
    })
})


In [7]:
def format_example(example):
    return {
        "input_text":  example["en"],
        "target_text": example["vi"]
    }

raw = raw.map(format_example)

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/19000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoTokenizer

model_name = "VietAI/envit5-translation"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_len = 256
max_target_len = 256

def tokenize_function(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=max_input_len,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target_text"],
        max_length=max_target_len,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw.map(tokenize_function, batched=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/19000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
!sudo apt-get install tree

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)





The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (91.8 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package tree.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [10]:
!tree -d /kaggle/input

/kaggle/input
├── competitions
│   └── ai-mathematical-olympiad-progress-prize-3
│       └── kaggle_evaluation
│           └── core
│               └── generated
└── datasets
    └── ctrungnguyn123
        └── envit5-lora-model-10-epoch

8 directories


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
!find /kaggle/input -name "adapter_config.json"

/kaggle/input/datasets/ctrungnguyn123/envit5-lora-model-10-epoch/adapter_config.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [12]:
!pip install -U torchao

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [13]:
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/input/datasets/ctrungnguyn123/envit5-lora-model-10-epoch"
)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(50048, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(50048, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [14]:
print(raw["test"]["input_text"][1])

The factors that reduced FEV1 is age, working with chemicals (p = 0.04) working in rubber plantation (p = 0, 049) and smoking.


In [15]:
def generate_text(sample):
    inputs = tokenizer(
        sample["input_text"],
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=4
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [16]:
!pip install tqdm bert-score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00


In [17]:
from tqdm import tqdm
import time

predictions = []
references = []

start_time = time.time()
for sample in tqdm(raw["test"], desc="Generating", unit="sample"):
    pred = generate_text(sample)
    predictions.append(pred)
    references.append(sample["vi"])

end_time = time.time()

print(f"\n⏱ Total time: {end_time - start_time:.2f} seconds")
print(f"⚡ Avg/sample: {(end_time - start_time)/len(predictions):.4f} sec")

Generating: 100%|██████████| 1000/1000 [22:06<00:00,  1.33s/sample]


⏱ Total time: 1326.39 seconds
⚡ Avg/sample: 1.3264 sec


In [18]:
import evaluate
import numpy as np

# ===== LOAD METRICS =====
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
ter = evaluate.load("ter")

print("\n🔍 Computing metrics...\n")

# ===== BLEU (cần list[list]) =====
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)["bleu"]


# ===== METEOR =====
meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)["meteor"]

# ===== TER =====
ter_score = ter.compute(
    predictions=predictions,
    references=references
)["score"]



# ===== PRINT GỌN =====
print("===== FINAL RESULTS =====")
print(f"BLEU          : {bleu_score:.4f}")
print(f"METEOR        : {meteor_score:.4f}")
print(f"TER (↓ better): {ter_score:.4f}")

2026-04-28 08:02:08.786667: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777363329.040702      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777363329.115446      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777363329.689699      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777363329.689747      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777363329.689750      23 computation_placer.cc:177] computation placer alr

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...



🔍 Computing metrics...

===== FINAL RESULTS =====
BLEU          : 0.4418
METEOR        : 0.6981
TER (↓ better): 51.8610
